<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Fast enough on a CPU

**[Fast enough on a CPU](https://learning.nextia-ai.com/courses/vision/m08/fast-enough-on-a-cpu/)** · Vision AI: See, Count and Inspect · Module 8, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure a vision model's time per photo and its accuracy after each change: a smaller image, a batch, export to ONNX and 8-bit numbers (int8).

| | |
|---|---|
| **Time** | About 40 minutes, of which about 2 minutes of computing |
| **Needs** | An internet connection for the setup cells and the ResNet-18 weights (47 MB). No account and no key. |
| **You should know** | Embeddings and linear probes, from [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 with PyTorch 2.15.0, timm 1.0.31, onnx 1.23.2 and ONNX Runtime 1.31.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/fast-enough-on-a-cpu/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M08-L02-fast-enough-on-a-cpu/fast-enough-on-a-cpu-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, torch 2.15.0, torchvision 0.29.2, timm 1.0.31, onnx 1.23.2, onnxruntime 1.31.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2 torch==2.15.0 torchvision==0.29.2 timm==1.0.31 onnx==1.23.2 onnxruntime==1.31.1
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)
import torch; print("torch", torch.__version__)
import torchvision; print("torchvision", torchvision.__version__)
import timm; print("timm", timm.__version__)
import onnx; print("onnx", onnx.__version__)
import onnxruntime; print("onnxruntime", onnxruntime.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 15 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The model and two measures

We use ResNet-18 without its last layer: it turns a photo into 512 numbers. A balanced logistic regression on those numbers names the product (the catalogue's eight classes). We measure two things after every change: the **accuracy** on the 160 test photos and the **time per photo** on one thread.

The times depend on your computer. Compare them with each other, not with the lesson's numbers.

In [ ]:
import time, torch, timm
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
# ResNet-18 trained on ImageNet (timm, Apache-2.0 weights), at a fixed revision, without its last layer.
weights = hf_hub_download("timm/resnet18.a1_in1k", "model.safetensors", revision="491b427b45c94c7fb0e78b5474cc919aff584bbf")
model = timm.create_model("resnet18.a1_in1k", pretrained=True, num_classes=0,
                          pretrained_cfg_overlay={"file": weights}).eval()

catalogue = pd.read_csv("data/catalogue/manifest.csv")
train = catalogue[catalogue["split"] == "train"].reset_index(drop=True)
test = catalogue[catalogue["split"] == "test"].reset_index(drop=True)
photos = {f: Image.open("data/" + f).convert("RGB") for f in pd.concat([train, test])["file"]}
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def prepare(img, size=224):
    """One photo as the network wants it: size x size pixels, scaled, shape (1, 3, size, size)."""
    a = np.asarray(img.resize((size, size), Image.BILINEAR), np.float32) / 255
    return ((a - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)

def pytorch(x):
    with torch.no_grad():
        return model(torch.from_numpy(x)).numpy()

def accuracy(run, size=224):
    """Embed every photo with `run`, fit a balanced logistic regression on train, return the accuracy on test."""
    def embed(rows):
        return np.vstack([run(np.vstack([prepare(photos[f], size) for f in rows["file"][i:i + 32]]))
                          for i in range(0, len(rows), 32)])
    clf = LogisticRegression(C=1.0, max_iter=5000, class_weight="balanced").fit(embed(train), train["label"])
    return round(float((clf.predict(embed(test)) == test["label"]).mean()), 3)

def ms_per_photo(run, size=224, batch=1, threads=1, photos_to_time=40):
    """Milliseconds per photo: one warm-up call, then enough calls for about 40 photos."""
    torch.set_num_threads(threads)
    x = np.vstack([prepare(photos[f], size) for f in test["file"][:batch]])
    run(x)
    start, done = time.perf_counter(), 0
    while done < photos_to_time:
        run(x)
        done += batch
    torch.set_num_threads(4)
    return round(1000 * (time.perf_counter() - start) / done, 1)

print("accuracy, PyTorch, 224 pixels:", accuracy(pytorch))
print("ms a photo, one at a time, 1 thread:", ms_per_photo(pytorch))

> **Check.** The accuracy is 0.85. On the course's laptop (Apple M4), one photo took 10.2 ms.

## 2. A smaller image

A photo of 160 × 160 pixels has half the pixels of 224 × 224, so the network does about half the work.

> **Predict.** Does the accuracy fall a lot, a little, or not at all?

In [ ]:
print("accuracy, 160 pixels:", accuracy(pytorch, 160))
print("ms a photo, 160 pixels:", ms_per_photo(pytorch, 160))

> **Check.** 0.838: one or two photos fewer than at 224 (one photo is 0.6 points). The time falls by about a third.

## 3. A batch of 16

A batch sends 16 photos in one call. It helps when many photos wait at the same time, and when the computer has many threads to share the work.

In [ ]:
print("ms a photo, batch of 16, 1 thread:", ms_per_photo(pytorch, batch=16))
print("ms a photo, one at a time, 4 threads:", ms_per_photo(pytorch, threads=4))

> **Check.** On the course's laptop, a batch of 16 on one thread was **slower** per photo (16.7 ms against 10.2). More threads helped more (7.6 ms). Measure on the computer that will run the service: a rule of thumb can be wrong.

## 4. Export to ONNX

ONNX is a file format for a trained network. ONNX Runtime runs the file without PyTorch: the service then needs a 20 MB library instead of PyTorch.

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, quantize_static

torch.onnx.export(model, (torch.randn(1, 3, 224, 224),), "resnet18.onnx", input_names=["image"], output_names=["features"],
                  dynamic_axes={"image": {0: "batch"}, "features": {0: "batch"}}, opset_version=18, dynamo=False)

def onnx_runner(path, threads):
    options = ort.SessionOptions()
    options.intra_op_num_threads = threads
    session = ort.InferenceSession(path, options, providers=["CPUExecutionProvider"])
    return lambda x: session.run(None, {"image": x})[0]

fp32 = onnx_runner("resnet18.onnx", threads=1)
print("ONNX file:", round(Path("resnet18.onnx").stat().st_size / 1e6, 1), "MB")
print("accuracy, ONNX fp32:", accuracy(onnx_runner("resnet18.onnx", 4)))
print("ms a photo, ONNX fp32, 1 thread:", ms_per_photo(fp32))

> **Check.** The same accuracy, 0.85: the exported file computes the same numbers. On the course's laptop it was **slower** on one thread (17.1 ms against 10.2). Export alone is not a speed-up; it is a lighter way to ship.

## 5. int8: 8-bit numbers

Quantisation stores the weights, and computes, with whole numbers from −128 to 127 instead of 32-bit decimals. Each layer keeps a scale to turn them back. The file is about four times smaller. ONNX Runtime prints a warning about "pre-processing": it is harmless here.

In [ ]:
class Calibration(CalibrationDataReader):
    """64 training photos: ONNX Runtime looks at the range of the numbers inside the network on them."""
    def __init__(self):
        self.photos = iter([{"image": prepare(photos[f])} for f in train["file"][:64]])
    def get_next(self):
        return next(self.photos, None)

quantize_static("resnet18.onnx", "resnet18_int8.onnx", Calibration(), quant_format=QuantFormat.QDQ, per_channel=True)

int8 = onnx_runner("resnet18_int8.onnx", threads=1)
print("int8 file:", round(Path("resnet18_int8.onnx").stat().st_size / 1e6, 1), "MB")
print("accuracy, ONNX int8:", accuracy(onnx_runner("resnet18_int8.onnx", 4)))
print("ms a photo, ONNX int8, 1 thread:", ms_per_photo(int8))

> **Check.** On the course's laptop: 11.3 MB, accuracy 0.856, 8.4 ms. The accuracy did not fall: one photo more is right, which is noise. Always measure the accuracy again after int8: on another model or other photos it can fall.

## 6. Your turn: an even smaller image

> **Your turn.** Put the accuracy of the PyTorch model at **128 × 128** pixels in `acc_128` (a number). Predict first: closer to 0.838 or to 0.80?

In [ ]:
acc_128 = ...

In [ ]:
expect('acc_128', acc_128, 0.838, tolerance=0.006)

## 7. Write the decision down

Fill in the table for Amira, with your own times: the change, the accuracy, the time per photo, the file size, and what the service must install. The course's choice for the inspector was ONNX int8 at 224 pixels: it needs only ONNX Runtime and NumPy, and it keeps the accuracy.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m08/fast-enough-on-a-cpu/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The image service and its review queue](https://learning.nextia-ai.com/courses/vision/m08/the-image-service-and-its-review-queue/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m08/fast-enough-on-a-cpu/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.